# Possession Lens — latest local notebook
Authors: Hrishikesh Karmalkar, Basel, Omar, Michail and Paul.

Run from top to bottom. The default run reads public external aggregate reports and runs local checks. Training, import and competition inference require explicit switches. No competition data, video, labels or predictions are uploaded. Clear all outputs before sharing.

The target is **possession-spell F1**. The default production checkpoint remains `rnn-boundary-001.npz` until a defensible promotion decision. A higher score on partial provider labels alone does not establish challenge accuracy.


In [ ]:
from pathlib import Path
import sys, json, subprocess, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "controlswitch").is_dir()), None)
assert ROOT is not None, "Open this notebook within the Possession Lens folder."
sys.path.insert(0, str(ROOT))
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
import numpy as np
import pandas as pd
from IPython.display import display, FileLink
print("Project:", ROOT.name, "| Python:", sys.version.split()[0])
PRODUCTION_MODEL = ROOT / "models/rnn-boundary-001.npz"
EXPERIMENTAL_MODEL = ROOT / "models/rnn-europe-context-001-experimental.npz"
MODEL = PRODUCTION_MODEL  # Full-match default; experimental weights are H/A-only transfer evidence.
EUROPE_ROOT = ROOT / "data/external/skillcorner-europe/raw"
EUROPE_REPORT = ROOT / "reports/external/rnn-europe-001-report.json"
RUN_IMPORT = False
RUN_TRAINING = False
RUN_INFERENCE = False
RUN_TESTS = True


## Latest: audited control recovery, interaction RNN and verified events

Named-control recovery now uses same contiguous provider track IDs bracketed by the same roster ID, plus short same-owner missing-ball gaps with continuous tracking and no observed nearby competitor. Training selected the limits; withheld PSG–Marseille observations gated them at >=99.5% proxy precision. These are automatic-observation/proximity references, **not independent human contact or identity truth**. Direct and inferred control remain separate; unresolved time stays in the denominator.

Natural coverage gains are modest: about **0.02–0.28 percentage points per SkillCorner team**. Long occlusions remain unresolved. The original 2m radius was not enlarged, existing direct named attributions are preserved, and no anonymous name is guessed.

A new 130-feature interaction-context RNN was trained from scratch on the existing five matches and selected on the existing validation match only. **Validation spell F1 improved 0.5042→0.5126**, with pooled misses 337→330, possible merges 72→68, false positives unchanged at 135 and short pulses 3→0. Repeated validation can overfit. **There is no new test score**; the gallery keeps context-001 and its last frozen fresh-test result of 0.6286. The 0.8 target is not reached.

Verified event stats are now available for **Bochum–Leverkusen** from its CC BY 4.0 Sportec feed, using exact provider PersonId roster joins. Completed passes include crosses/set pieces. Shots/on-target use explicit provider event outcomes. Touches remain unavailable because the feed does not establish every physical touch. Other gallery matches show unavailable event counters, never tracking-derived guesses. The event timeline supports goals, substitutions, shots and completed passes, with half filters.


In [ ]:
control_audit = json.loads((ROOT / "reports/external/control-recovery-001-report.json").read_text())
for kind in ("identity", "ball"):
    print(kind, "enabled:", control_audit[kind]["enabled"], "withheld-observation validation:", control_audit[kind]["validation"])
coverage = json.loads((ROOT / "reports/external/control-recovery-001-coverage.json").read_text())
display(pd.DataFrame(coverage["rows"])[["match_id", "team", "direct_percent", "new_mapped_percent", "gain_percentage_points"]].round(3))
interaction = json.loads((ROOT / "reports/external/rnn-interaction-context-001-report.json").read_text())
print("Reference validation spell F1:", interaction["reference_validation"]["selection_spell_f1"])
print("Selected validation spell F1:", interaction["selected"]["selection_spell_f1"] if interaction["selected"] else None)
print("Fresh-test results:", interaction["fresh_test_results"])
print("Promoted to default:", interaction["promoted_to_default"])
events = json.loads((ROOT / "examples/verified-events-J03WN1.json").read_text())
display(pd.DataFrame(events["teams"]).fillna("Not available"))
print(events["source"])
print("Event definitions:", events["definitions"])


In [ ]:
RUN_INTERACTION_TRAINING = False
RUN_CONTROL_RECOVERY_AUDIT = False
if RUN_INTERACTION_TRAINING or RUN_CONTROL_RECOVERY_AUDIT:
    from datetime import datetime
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    raw = ROOT / "data/external/skillcorner-europe/raw"
    if RUN_INTERACTION_TRAINING:
        from controlswitch.interaction_context import run as train_interactions
        train_interactions(raw, ROOT / "models/rnn-europe-context-001-experimental.npz",
            ROOT / "models/rnn-europe-001-experimental.npz", ROOT / "local" / ("interaction-" + stamp), iterations=12)
    if RUN_CONTROL_RECOVERY_AUDIT:
        from controlswitch.control_recovery_audit import run as audit_control
        audit_control(raw, ROOT / "models/rnn-europe-context-001-experimental.npz",
            ROOT / "local" / ("control-audit-" + stamp))
else:
    print("Retraining and recovery audits disabled. The notebook displays saved external reports.")


## Latest update: ten named examples and fresh-data status

The offline website now includes a match picker for all nine historical SkillCorner matches plus the existing Bochum–Leverkusen example. Every displayed player is joined to a provider roster. Anonymous nearest tracks and missing evidence remain unresolved, never renamed. Impact means **estimated proximity control involvement**, not confirmed touches, passes or causal attacking value. Mapped control covers only part of each team's possession; the unresolved portion remains visible and in the denominator.

The nine-match release is fully used. An open-access search found no further anonymously downloadable top-five-league dataset with verified continuous tracking, possession supervision and suitable rights. Premier League 2024/25 and 2025/26 collections are free but gated with research/education conditions; Driblab's current published repository does not establish an available labelled, licensed download. Event-only files cannot supply continuous tracking truth. See `docs/data-sources/OPEN_ACCESS_AUDIT_2026-11-12.md`.

A finer decoder search used development match 2269 only. No setting improved validation while preserving the miss/merge and false-switch safeguards, so penalty 48 and the previous context checkpoint remain unchanged. **No new fresh-test result is claimed.** The last frozen independent-match mean spell F1 remains 0.6286, below 0.8. Viewing the existing gallery matches is not a fresh evaluation.

The user selected **anonymous downloads only**, so gated sources are excluded. Cologne–Bayern tracking is also local, but its provider roster is unavailable; it is withheld from the named gallery rather than given guessed names.


In [ ]:
library_path = ROOT / "examples/match-library.json"
if library_path.exists():
    library = json.loads(library_path.read_text())
    display(pd.DataFrame([{"match": m["title"], "league": m.get("league"),
        "date": m.get("date"), "mapped_players": sum(r["person_id"] != "UNASSIGNED" for r in m["named_players"]["rows"])}
        for m in library["matches"]]))
    selected_example = next(m for m in library["matches"] if m.get("example_id") == "skillcorner-2841")
    display(pd.DataFrame([r for r in selected_example["named_players"]["rows"] if r["person_id"] != "UNASSIGNED"])
        [["team", "player_name", "associated_control_seconds", "percent_of_team_possession", "associated_bouts"]].round(3))
    print("All displayed people are provider-mapped. Percentages include unresolved control in the denominator.")
    if (ROOT / "dashboard.html").exists(): display(FileLink(str(ROOT / "dashboard.html")))
refinement_path = ROOT / "reports/external/rnn-decoder-refinement-001-report.json"
if refinement_path.exists():
    refinement = json.loads(refinement_path.read_text())
    print("Decoder improved validation:", refinement["improved_validation"])
    print("Fresh test result:", refinement["fresh_test_results"])
    print("Retained reference validation F1:", refinement["reference_validation"]["selection_spell_f1"])


In [ ]:
RUN_REBUILD_NAMED_LIBRARY = False
if RUN_REBUILD_NAMED_LIBRARY:
    from controlswitch.example_gallery import build as build_named_library
    from controlswitch.dashboard import export_html
    build_named_library(ROOT / "data/external/skillcorner-europe/raw",
        ROOT / "models/rnn-europe-context-001-experimental.npz", ROOT / "examples/matches")
    export_html(ROOT / "dashboard.html")
else:
    print("Gallery rebuild disabled; the bundled ten-match website works offline.")


## 1. Five European matches
Training: Manchester City–Liverpool (4039), Dortmund–Bayern (3749), Juventus–Inter (3518).
Validation: PSG–Marseille (2269). Reserved test: Real Madrid–Barcelona (3442).

Source: official SkillCorner open data, MIT licence, pinned revision `1ed50b775f93075c2a6d00d0d795f13b7b89709e`. Preserve attribution. Inputs are ignored by Git. Provider-generated home/away possession is weak supervision; null is **unknown**, never dead ball. Adjacent observed tracking is interpolated from 10 Hz to the common 25 Hz interface; gaps are not filled.

Do not open test labels during development. The experiment freezes the checkpoint and decoder before test loading. After the first result, this match is a viewed test; rerunning it reproduces evidence and cannot create another fresh test.


In [ ]:
from controlswitch.skillcorner_legacy import SPLIT
print("Frozen split:", SPLIT)
manifest_path = EUROPE_ROOT / "import-manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("Verified imported files:", len(manifest["files"]))
else:
    print("External archive not imported locally yet; source-only checkouts do not include raw data.")


## 2. Import the already downloaded archive (optional)
Download the pinned official repository ZIP in your browser. This cell extracts only the five selected matches, verifies expected file sizes and Git blob hashes, and preserves the licence. It does not parse test possession labels. Existing destinations are protected from overwrite.


In [ ]:
ARCHIVE = Path.home() / "Downloads/opendata-1ed50b775f93075c2a6d00d0d795f13b7b89709e.zip"
if RUN_IMPORT:
    from controlswitch.skillcorner_legacy import import_archive
    catalog = json.loads((ROOT / "docs/external/top-five-europe-catalog.json").read_text())
    imported = import_archive(ARCHIVE, catalog, EUROPE_ROOT)
    print("Verified files:", len(imported["files"]))
else:
    print("Import disabled; existing local data remains available.")


## Latest: expanded training and temporal-context RNN

Training now includes **five** top-five-league matches: Manchester City–Liverpool (4039), Dortmund–Bayern (3749), Juventus–Inter (3518), **Inter–Juventus (2068)** and **Liverpool–Manchester City (2440)**. PSG–Marseille (2269) remains validation. Reserved fresh test: **Barcelona–Real Madrid, 18 December 2019 (2841)**. Previously scored tests 3442 and 2417 are excluded from development.

Two RNNs are trained from scratch with the same expanded pool: instantaneous features, and 120 multiscale features containing observations, means over ±0.48 and ±2 seconds, and a movement contrast. The latter uses surrounding tracking to distinguish a brief possession from noise. It is **offline analysis**, not live forecasting. Neither loss upweighting nor the rejected boundary head is reused.

Each has one layer of 24 units and up to 16 sampled iterations, with dropout, L2 and gradient clipping. Settings are selected by validation possession-spell F1 with false-positive and short-pulse limits. Both family checkpoints, their decoder settings and the baseline are frozen before test labels are opened. Test ablations are reported for diagnosis and cannot change the validation-selected model.

Labels are partial and automatic; missing labels stay unknown. Different matches' F1 values cannot establish an improvement: compare models on the same held-out match. The 0.8–0.9 target remains unproven.


In [ ]:
CONTEXT_REPORT = ROOT / "reports/external/rnn-europe-context-001-report.json"
if CONTEXT_REPORT.exists():
    context_report = json.loads(CONTEXT_REPORT.read_text())
    print("Fresh test: Barcelona–Real Madrid, La Liga, 18 December 2019")
    print("Validation-selected family:", context_report["frozen"]["family"])
    print("Passes validation false-positive guard:", context_report["frozen"]["passes_false_positive_guard"])
    print("Improves validation F1:", context_report["frozen"]["validation_improved"])
    display(pd.DataFrame([{"model": name,
        "clean_spell_F1": r["clean"]["spell_f1_mean"],
        "dropout_spell_F1": r["masked"]["spell_f1_mean"],
        "mean_spell_F1": r["selection_spell_f1"]}
        for name, r in context_report["test_results"].items()]).round(4))
    print("Selected candidate beats previous European RNN:", context_report["candidate_beats_previous_european"])
    print(context_report["decision"])
    print("Diagnostic changes vs previous (negative means fewer):", context_report["diagnostic_changes_vs_previous"])
    diagnostic_rows = []
    for model, conditions in context_report["test_diagnostics"].items():
        for condition, d in conditions.items():
            diagnostic_rows.append({"model": model, "condition": condition,
                "missed_spells_IoU_0.5": d["missed_reference_spells_at_iou_0_5"],
                "false_positives_IoU_0.5": d["false_positive_spells_at_iou_0_5"],
                "possible_merges": d["predictions_overlapping_multiple_reference_spells"],
                "pulses_under_0.4s": d["predicted_spells_under_0_4_seconds"]})
    display(pd.DataFrame(diagnostic_rows))
    display(pd.DataFrame(context_report["test_audit"]))
    for limitation in context_report["limitations"]:
        print("•", limitation)
else:
    print("Context experiment is not complete; no test improvement is assumed.")


In [ ]:
for family in ("instantaneous", "context"):
    learning_path = ROOT / f"reports/external/rnn-europe-context-001-{family}-learning.json"
    if learning_path.exists():
        print("Learning history:", family)
        display(pd.DataFrame(json.loads(learning_path.read_text())).round(4))
print("Iterations sample at most 1,024 windows; they are not full epochs.")


### Optional: import the three additions and reproduce training

Source-only GitHub checkouts do not include raw data. The original five-match importer appears above. The optional cell below verifies and imports only 2068, 2440 and 2841 from the pinned official ZIP, preserving the test without parsing labels. Existing files are protected from overwrite. Reproducing a previously viewed test is not fresh evaluation.


In [ ]:
IMPORT_CONTEXT_ADDITIONS = False
if IMPORT_CONTEXT_ADDITIONS:
    from controlswitch.context_data import import_additions
    context_import = import_additions(ARCHIVE, EUROPE_ROOT)
    print("Verified new files:", len(context_import["files"]))
else:
    print("Context-data import disabled. Existing local data stays available.")

RUN_CONTEXT_TRAINING = False
if RUN_CONTEXT_TRAINING:
    from datetime import datetime
    from controlswitch.europe_context import run as train_context
    context_out = ROOT / "local" / ("context-reproduction-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    context_out.mkdir(parents=True)
    protocol = json.loads((ROOT / "reports/external/rnn-europe-context-001-preregistered-protocol.json").read_text())
    protocol["test_previously_parsed"] = True
    protocol["execution_mode"] = "reproduction_on_viewed_test"
    (context_out / "preregistered-protocol.json").write_text(json.dumps(protocol, indent=2) + "\n")
    train_context(EUROPE_ROOT, context_out,
        ROOT / "models/rnn-europe-001-experimental.npz", MODEL, iterations=16)
else:
    print("Expanded-data training disabled; existing aggregate results are displayed above.")


## 3. Latest five-league experiment
A new candidate is trained **from scratch** on the three selected top-five-league matches. One- and two-layer RNNs use 24 units per layer, dropout, L2 and gradient clipping. Up to 12 sampled iterations are compared. Checkpoints and decoder penalties are selected by mean clean/dropout **validation spell F1**, rather than training loss. Training/validation loss remains an overfitting diagnostic.

All models receive the same tracking and label mask. No dead-ball targets are invented. Reported F1 is conditional on provider-labelled runs, not independently annotated full-match possession.


In [ ]:
if EUROPE_REPORT.exists():
    europe = json.loads(EUROPE_REPORT.read_text())
    rows = []
    for name, result in europe["test_results"].items():
        rows.append({"model": name, "clean_spell_F1": result["clean"]["spell_f1_mean"],
                     "dropout_spell_F1": result["masked"]["spell_f1_mean"],
                     "mean_spell_F1": result["selection_spell_f1"]})
    display(pd.DataFrame(rows).round(4))
    print("Candidate beats current on this conditional test:", europe["candidate_beats_current"])
    print("Promoted to default:", europe["promoted_to_default"])
    print("Selected configuration:", europe["frozen"])
    display(pd.DataFrame(europe["test_audit"]))
    for limitation in europe["limitations"]:
        print("•", limitation)
else:
    print("European experiment report is not available yet. No test score is assumed.")


In [ ]:
learning_path = ROOT / "reports/external/rnn-europe-001-validation-trials.json"
if learning_path.exists():
    learning = pd.DataFrame(json.loads(learning_path.read_text()))
    best_per_iteration = learning.sort_values("score", ascending=False).drop_duplicates(["layers", "iteration"]).sort_values(["layers", "iteration"])
    display(best_per_iteration.round(4))
    print("A sampled iteration uses at most 1,024 windows of 128 frames; it is not a full epoch.")


## Latest: recovering missed and merged possessions

The latest experiment keeps the original three training matches and PSG–Marseille validation. Previously scored Real Madrid–Barcelona is **excluded from development**. A different match, Bayern–Dortmund on 9 November 2019 (2417), was reserved before training and opened only after freezing the selected model.

The candidate uses spell-balanced loss to emphasise short possessions, plus an optional trained boundary head to reduce switching costs where trajectory/geometry supports a change. Unknown-censored spells are not treated as complete short possessions. Unknown frames carry zero loss; no dead-ball labels are invented. Model selection maximises validation spell F1 subject to false-positive and short-pulse limits. Extra layers and more iterations are not assumed to improve F1.

The fresh test shares clubs with training. Provider labels cover only observed H/A runs, so results remain conditional transfer evidence, not full-match challenge accuracy. Once scored, 2417 is a viewed test and must not guide further tuning.


In [ ]:
RECOVERY_REPORT = ROOT / "reports/external/rnn-europe-recovery-001-report.json"
if RECOVERY_REPORT.exists():
    recovery = json.loads(RECOVERY_REPORT.read_text())
    display(pd.DataFrame([{"model": name,
        "clean_spell_F1": r["clean"]["spell_f1_mean"],
        "dropout_spell_F1": r["masked"]["spell_f1_mean"],
        "mean_spell_F1": r["selection_spell_f1"]}
        for name, r in recovery["test_results"].items()]).round(4))
    print("Test: Bayern–Dortmund, Bundesliga, 9 November 2019")
    print("Selected configuration:", recovery["frozen"])
    print("Recovery changes accepted:", recovery["recovery_changes_accepted"])
    print(recovery["decision"])
    print("Candidate beats previous European RNN:", recovery["candidate_beats_previous_european"])
    print("Promoted to production default:", recovery["promoted_to_default"])
    diagnostic_rows = []
    for model, conditions in recovery["test_diagnostics"].items():
        for condition, d in conditions.items():
            diagnostic_rows.append({"model": model, "condition": condition,
                "missed_spells_IoU_0.5": d["missed_reference_spells_at_iou_0_5"],
                "false_positive_spells_IoU_0.5": d["false_positive_spells_at_iou_0_5"],
                "merge_overlap_count": d["predictions_overlapping_multiple_reference_spells"],
                "pulses_under_0.4s": d["predicted_spells_under_0_4_seconds"]})
    display(pd.DataFrame(diagnostic_rows))
    print("Merge overlap counts are descriptive geometry, not confirmed event annotations.")
    display(pd.DataFrame(recovery["test_audit"]))
    for limitation in recovery["limitations"]:
        print("•", limitation)
else:
    print("Recovery experiment report not available yet. No improvement is assumed.")


In [ ]:
recovery_learning = ROOT / "reports/external/rnn-europe-recovery-001-learning.json"
if recovery_learning.exists():
    display(pd.DataFrame(json.loads(recovery_learning.read_text())).round(4))
    print("Training objective is weighted; train/validation probe CE is unweighted for comparison.")
    print("Checkpoint selection uses validation spell F1 with false-positive limits, not minimum training loss.")


In [ ]:
IMPORT_FRESH_TEST = False
if IMPORT_FRESH_TEST:
    from controlswitch.skillcorner_fresh_import import import_fresh_test
    imported_fresh = import_fresh_test(ARCHIVE, EUROPE_ROOT)
    print("Reserved match imported without parsing tracking:", len(imported_fresh["files"]), "verified files")
else:
    print("Sixth-match import disabled. Enable only if 2417 is not already imported.")


### Optional: reproduce the recovery experiment
Leave disabled for the judge walkthrough. This is a lengthy local retraining run. Reproduction uses the already viewed test and cannot provide new independent evidence. It does not replace the production checkpoint. All raw files must already be imported locally.


In [ ]:
RUN_RECOVERY_TRAINING = False
if RUN_RECOVERY_TRAINING:
    from datetime import datetime
    from controlswitch.europe_recovery import run as recover_spells
    recovery_out = ROOT / "local" / ("recovery-reproduction-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    recovery_out.mkdir(parents=True)
    protocol = json.loads((ROOT / "reports/external/rnn-europe-recovery-001-preregistered-protocol.json").read_text())
    protocol["test_previously_parsed"] = True
    protocol["execution_mode"] = "reproduction_on_viewed_test"
    (recovery_out / "preregistered-protocol.json").write_text(json.dumps(protocol, indent=2) + "\n")
    recover_spells(EUROPE_ROOT, recovery_out,
        ROOT / "models/rnn-europe-001-experimental.npz", MODEL, iterations=12)
else:
    print("Recovery retraining disabled. Existing results are shown above.")


## 4. Reproduce external-only training (optional, potentially lengthy)
Leave disabled for the judge walkthrough. It creates a new local experiment and does not replace the default checkpoint. The held-out result becomes reproduction of an already viewed test. Never reuse it for model selection or tune after reading the result.


In [ ]:
if RUN_TRAINING:
    from datetime import datetime
    from controlswitch.skillcorner_europe import run
    reproduction_out = ROOT / "local" / ("europe-reproduction-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    run(EUROPE_ROOT, reproduction_out, MODEL, iterations=12)
else:
    print("Training disabled. Default checkpoint:", MODEL.name)


## 5. Earlier independent test evidence
The current model was retained after the J03WR9 test. These historical second-division results remain comparison evidence; they are excluded from the new European first-division training pool. Historical test matches are already viewed and cannot be reused as fresh tests.


In [ ]:
historical_path = ROOT / "reports/external/rnn-fresh-J03WR9-001-report.json"
if historical_path.exists():
    historical = json.loads(historical_path.read_text())
    display(pd.DataFrame([{"model": name,
        "clean_spell_F1": r["clean"]["spell_f1_mean"],
        "dropout_spell_F1": r["masked"]["spell_f1_mean"],
        "mean_spell_F1": r["mean_clean_masked_spell_f1"]}
        for name, r in historical["fresh_match_results"].items()]).round(4))


## 6. Judge experience and player statistics
Open `dashboard.html` from the project folder, or use `Open Possession Lens.command`. It starts the **named external Barcelona–Real Madrid example**. The Match, Players and Model evidence tabs show team possession, provider-mapped names and model evidence. Challenge tracking slots have no permitted name mapping and remain provisional tracklets; they cannot support verified individual-player claims.

Team percentages exclude dead and unknown frames. Individual control shares must preserve unassigned time and are distinct from total team possession. No manual challenge player mapping or substitution annotations are permitted.


In [ ]:
if (ROOT / "dashboard.html").exists():
    display(FileLink(str(ROOT / "dashboard.html")))
print("Named external demo by default; private challenge outputs remain local.")


## 7. Apply the frozen model to the challenge locally (optional)
This runs inference only, with no training, tuning, calibration or video comparison. All outputs remain in the ignored `local` directory. No private statistics are displayed by this notebook. Do not publish notebook outputs or private match files.


In [ ]:
DATA_ROOT = ROOT / "data/raw/provided"
if RUN_INFERENCE:
    from datetime import datetime
    prediction_out = ROOT / "local" / ("notebook-inference-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    subprocess.run([sys.executable, "-m", "possession_lens", "rnn-predict", "--data-root", str(DATA_ROOT), "--model", str(MODEL), "--out", str(prediction_out)], cwd=ROOT, check=True)
    print("Local inference completed. Keep its results private.")
else:
    print("Challenge inference disabled.")


## Short-spell weighted context RNN — validation-only experiment
Two preregistered variants use the same five external training matches and PSG–Marseille validation match: 24 hidden units versus two layers of 24. Short, uncensored labelled spells receive higher loss weights. Unknown labels remain excluded. Dropout, weight decay, team symmetry, simulated ball dropout and fixed decoder penalties are retained. No viewed test or competition match is used.
Selection requires no increase in missed/merged possessions versus interaction-context-001, plus the original switching guard. Repeated validation risks overfitting; these results cannot establish generalisation or a fresh-test improvement. The gallery model remains unchanged.


In [ ]:
spell_path = ROOT / "reports/external/rnn-spell-context-002-report.json"
if spell_path.exists():
    spell_report = json.loads(spell_path.read_text())
    print("Validation only; default model not promoted.")
    display(pd.DataFrame([{ "model": name, "validation_spell_f1": value["selection_spell_f1"] } for name, value in [("baseline", spell_report["baseline_validation"]), ("previous interaction RNN", spell_report["reference_validation"])] + ([("selected weighted RNN", spell_report["selected"])] if spell_report["selected"] else [])]))
    display(pd.DataFrame([{ "hidden_sizes": str(v["hidden_sizes"]), "validation_spell_f1": v["selection_spell_f1"], "passed_all_guards": v["eligible"] } for v in spell_report.get("best_unrestricted_by_architecture", [])]))
else:
    print("Short-spell weighted experiment has no completed report in this checkout.")


In [ ]:
RUN_SPELL_CONTEXT_TRAINING = False
if RUN_SPELL_CONTEXT_TRAINING:
    from datetime import datetime
    from controlswitch.spell_context import run as train_spell_context
    out = ROOT / "local" / ("spell-context-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    train_spell_context(EUROPE_ROOT, ROOT / "models/archive/rnn-interaction-context-validation-only.npz",
        ROOT / "models/rnn-europe-001-experimental.npz", out, iterations=12)
else:
    print("Short-spell retraining disabled; saved validation report shown above.")


## Gated architectures — GRU, LSTM and boundary supervision
This optional experiment compares a one-layer GRU, two-layer bidirectional GRU, two-layer bidirectional LSTM, and bidirectional GRU with a supervised boundary head. It adds a continuous feature projection, layer normalisation, dropout, Xavier/orthogonal initialisation and norm clipping. Windowed backpropagation and inference use the same 256-frame core and 64 context frames on each side. Bidirectional results are offline.
Training uses the existing five external matches; validation uses 2269 only. No fresh test or competition data is loaded. The default/gallery model is unchanged. Scheduled sampling and text-generation RL are not applied because earlier labels and language tokens are never inputs. See `docs/external/GATED_RNN_003.md` for exact scope, limits and a standalone reproduction command.


In [ ]:
gated_path = ROOT / "reports/external/rnn-gated-003-report.json"
if gated_path.exists():
    gated = json.loads(gated_path.read_text())
    print("Validation only. Promoted to default:", gated["promoted_to_default"])
    display(pd.DataFrame([{ "variant": t["variant"], "iteration": t["iteration"], "validation_spell_f1": t["selection_spell_f1"], "eligible": t["eligible"], **t["guards"] } for t in gated["best_unrestricted_by_variant"]]))
    print("Previous interaction validation F1:", gated["reference_validation"]["selection_spell_f1"])
    print("Selected:", gated["selected"]["selection_spell_f1"] if gated["selected"] else "No candidate passed all guards")
    curves = ROOT / "reports/external/rnn-gated-003-learning.png"
    if curves.exists(): display(FileLink(str(curves)))
else:
    print("No completed gated experiment report in this checkout.")


In [ ]:
RUN_GATED_TRAINING = False
if RUN_GATED_TRAINING:
    import importlib.util
    assert importlib.util.find_spec("torch"), "Install requirements-gated.txt in a Python 3.11/3.12 environment; see the gated experiment guide."
    from datetime import datetime
    from controlswitch.gated_experiment import run as train_gated
    gated_out = ROOT / "local" / ("gated-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    train_gated(EUROPE_ROOT, ROOT / "models/archive/rnn-interaction-context-validation-only.npz", ROOT / "models/rnn-europe-001-experimental.npz", gated_out)
else:
    print("Gated retraining disabled. Existing models and website do not require PyTorch.")


## Decoder refinement and corrected short-spell diagnostics
The same gated weights with penalty20/strength1 improve reused-validation spell F1 from0.5527 to0.5554. Misses318→316; false positives115→112; merge overlaps66 unchanged. No promotion or fresh-test claim. All eight earlier short scored fragments were long predictions clipped by reference-label gaps; both full prediction sequences contain zero spells under0.4seconds. See `docs/external/GATED_DECODER_004.md`.
Additional data search: Driblab is a new lead, but data access, licensing and possession supervision remain unverified. The checked Premier League/Real Madrid collections are gated and excluded. See `docs/external/DATA_SEARCH_2026_11_13.md`.


In [ ]:
decoder_path = ROOT / "reports/external/rnn-gated-decoder-004-report.json"
if decoder_path.exists():
    decoder = json.loads(decoder_path.read_text())
    display(pd.DataFrame([{ "model": name, "validation_spell_f1": row["selection_spell_f1"] } for name, row in [("saved gated reference", decoder["reference_validation"]), ("refined decoder (validation only)", decoder["selected"])]]))
    print("No weight training, fresh test or default promotion. Reference-label clipping is not predicted flicker.")
RUN_DECODER_REPRODUCTION = False
if RUN_DECODER_REPRODUCTION:
    from datetime import datetime
    from controlswitch.gated_decoder import run as reproduce_decoder
    reproduce_decoder(EUROPE_ROOT, ROOT / "models/archive/rnn-gated-003-validation-only.npz", ROOT / "local" / ("decoder-" + datetime.now().strftime("%Y%m%d-%H%M%S")))


## 8. Local checks and sharing
Share source, the cleared notebook, dependencies, checkpoint, licence notices and aggregate external reports. Never upload challenge data, video, manually generated labels, prediction timelines, local manifests or raw external assets by accident. The source archive excludes `data`, `local`, `work` and `.git`.


In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=ROOT, check=True)
print("Notebook complete. Clear all outputs before committing or sharing.")
